# Projet de Statistique Appliquée

## Étude comparative de l'efficacité des hôpitaux selon leur statut

Projet d'économétrie appliqué s'appuyant sur les données publiques de la Statistique Annuelle des Établissements (SAE) et d'Hospidiag.

### Importation des bibliothèques

In [83]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

### Importations des données

Extraction des Outputs et Inputs de 2022, dans un premiers temps on s'intéresse aux services d'Urgence, de Médecine, Chirurgie et obstétrique (MCO), Soins de suite et rééducation (SSR), Psychiatrie, Unité de soins de longue durée (USLD), et séances (séances de dyalise, de chimiothérapie, etc).

In [84]:
SYGEN2022 = pd.read_csv("SAE/2022/SYGEN_2022r.csv", sep=";", encoding="latin-1")
Data = SYGEN2022[['FI','EFFSAL_TOT','EFFLIB_TOT','EFF_INFSANSSPE','EFF_INFAVECSPE','EFF_AID','EFF_DIR','EFF_DIRSOI','EFF_AUTADM','SEJHC_SSR','SEJHC_MCO','SEJHP_MCO','SEJ_HTP_TOT','VEN_HDJ_TOT','VEN_HDN_TOT','SEAN_HEMO_CENTRE','SEAN_CHIMIO','SEAN_RADIO']]

On extrait le statut des différents établissement depuis le fichier FINESS.xlsx obtenu sur le site SAE diffusion.
On crée une distinction entre les hôpitaux publics classiques qui serviront par la suite de référence et les grands hôpitaux publics, les CHU, en ajoutant une catégorie 'CHU' dans la colonne 'Statut'

In [85]:
FINESS = pd.read_excel("finess.xlsx")
FINESS = FINESS.rename(columns={"FINESS":"FI","Statut Juridique":"Statut"})

CHU = pd.read_excel("CHRU.xlsx")
CHU["FINESS"] = CHU["FINESS"].astype(object)

FINESS["Statut"][FINESS['Raison sociale'].isin(CHU['Raison sociale'])] = "CHU"
FINESS = FINESS.drop(FINESS.columns[[1,2,4]],axis=1)  #supression des colonnes inutiles

Data = Data.merge(FINESS, on="FI", how="left")

C:\Users\Alexandre\AppData\Local\Temp\ipykernel_21796\1451615058.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  FINESS["Statut"][FINESS['Raison sociale'].isin(CHU['Raison sociale'])] = "CHU"


In [86]:
print(Data.shape)
print(SYGEN2022.shape)

(3988, 19)
(3988, 208)


### Constructions des variables de la régression

Ajout des output des services d'urgences (nombre de passages) :

In [87]:
Urg2022 = pd.read_csv("SAE/2022/URGENCES2_2022r.csv", sep=";", encoding="latin-1")
Urg2022 = Urg2022[['FI','PASSU']]
Data = Data.merge(Urg2022, on="FI", how="left")

In [88]:
USLD2022 = pd.read_csv("SAE/2022/USLD_2022r.csv", sep=";", encoding="latin-1")
USLD2022 = USLD2022[['FI','ENT']]
Data = Data.merge(USLD2022, on="FI", how="left")

Total médecins (libéraux et saliariés confondus) : 

In [89]:
Data['MED'] = Data['EFFSAL_TOT'] + Data['EFFLIB_TOT'].fillna(0)
Data['MED'].describe()

count    3402.000000
mean       55.711934
std       115.955149
min         0.000000
25%         4.000000
50%        10.000000
75%        54.000000
max      1515.000000
Name: MED, dtype: float64

Total infirmiers (spécialisés et non-spécialisés) : 

In [90]:
Data['IDE'] = Data['EFF_INFSANSSPE'] + Data['EFF_INFAVECSPE'].fillna(0)
Data['IDE'].describe()

count    3600.000000
mean      111.296111
std       234.967032
min         0.000000
25%        11.000000
50%        25.000000
75%        83.000000
max      2090.000000
Name: IDE, dtype: float64

Total personnel administratif :

In [91]:
Data['ADMIN'] = Data['EFF_DIR'] + Data['EFF_AUTADM']
Data["ADMIN"].describe()

count    2667.000000
mean       62.586052
std       111.042866
min         0.000000
25%        10.000000
50%        19.000000
75%        60.000000
max       950.000000
Name: ADMIN, dtype: float64

In [92]:
Data['EFF_DIRSOI'].describe()

count    1050.000000
mean        1.177143
std         0.630532
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max         6.000000
Name: EFF_DIRSOI, dtype: float64

Total séjours MCO (hospitalisation partielle et complète confondu) :

In [93]:
Data['MCO'] = Data['SEJHP_MCO'] + Data['SEJHC_MCO']
Data['MCO'] = Data['MCO'].fillna(0)
Data['I_MCO'] = (Data['MCO'] > 0).astype(int)

In [94]:
Data['I_MCO']

0       1
1       1
2       1
3       0
4       1
       ..
4053    1
4054    0
4055    0
4056    0
4057    0
Name: I_MCO, Length: 4058, dtype: int32

Total séjours en psychiatrie (hospitalisations complètes, venues de jour et venues de nuit) :

In [95]:
Data['PSY'] = Data['SEJ_HTP_TOT'] + Data['VEN_HDJ_TOT'].fillna(0) + Data['VEN_HDN_TOT'].fillna(0)
Data['PSY'].describe()

count      541.000000
mean      7443.051756
std       7648.619662
min          0.000000
25%       1797.000000
50%       5215.000000
75%      10506.000000
max      47168.000000
Name: PSY, dtype: float64

In [96]:
Data['PSY'] = Data['PSY'].fillna(0)
Data['I_PSY'] = (Data['PSY'] > 0).astype(int)

In [97]:
Data["SEANCES"] = Data['SEAN_HEMO_CENTRE'].fillna(0) + Data['SEAN_CHIMIO'] + Data['SEAN_RADIO'].fillna(0)
Data["SEANCES"].describe()

count      795.000000
mean     10883.817610
std      15793.585328
min          0.000000
25%        697.000000
50%       3689.000000
75%      14192.000000
max      81352.000000
Name: SEANCES, dtype: float64

In [98]:
Data['SEANCES'] = Data['SEANCES'].fillna(0)
Data['I_SEANCES'] = (Data['SEANCES'] > 0).astype(int)

In [99]:
Data['PASSU'].describe()

count       693.000000
mean      31217.946609
std       19439.701167
min           0.000000
25%       16727.000000
50%       26546.000000
75%       40563.000000
max      113596.000000
Name: PASSU, dtype: float64

In [100]:
Data['PASSU'] = Data['PASSU'].fillna(0)
Data['I_URG'] = (Data['PASSU'] > 0).astype(int)

In [101]:
Data['PASSU'][Data['PASSU'] > 0]

0       42384.0
1       16727.0
13      22583.0
23      23369.0
24      38727.0
         ...   
3992    18143.0
3993    35983.0
3994    14081.0
3995    31213.0
4052    49595.0
Name: PASSU, Length: 692, dtype: float64

In [102]:
Data['ENT'].describe()

count    592.000000
mean      29.587838
std       20.884803
min        0.000000
25%       17.000000
50%       24.000000
75%       37.000000
max      167.000000
Name: ENT, dtype: float64

In [103]:
Data['ENT'] = Data['ENT'].fillna(0)
Data['I_USLD'] = (Data['ENT'] > 0).astype(int)

In [104]:
Data["SEJHC_SSR"].describe()

count    1847.000000
mean      472.886302
std       379.944830
min         0.000000
25%       220.000000
50%       377.000000
75%       632.500000
max      2755.000000
Name: SEJHC_SSR, dtype: float64

In [105]:
Data['SEJHC_SSR'] = Data['SEJHC_SSR'].fillna(0)
Data['I_SSR'] = (Data['SEJHC_SSR'] > 0).astype(int)

Passage des variables d'intérêt en log (En ajoutant +1 ppur éviter des valeurs négatives/-infini) :

In [106]:
Data["lURG"] = np.log(Data["PASSU"]+1)
Data["lMCO"] = np.log(Data["MCO"]+1)
Data["lPSY"] = np.log(Data["PSY"]+1)
Data["lSSR"] = np.log(Data["SEJHC_SSR"]+1)
Data["lSEANCES"] = np.log(Data["SEANCES"]+1)
Data['lUSLD'] = np.log(Data['ENT']+1)

Data["lMED"] = np.log(Data["MED"]+1)
Data["lIDE"] = np.log(Data["IDE"]+1)
Data["lAID"] = np.log(Data["EFF_AID"]+1)
Data["lADMIN"] = np.log(Data["ADMIN"]+1)

In [107]:
Data["lMED"].describe()

count    3402.000000
mean        2.830215
std         1.497048
min         0.000000
25%         1.609438
50%         2.397895
75%         4.007333
max         7.323831
Name: lMED, dtype: float64

### Ajout de variables de contrôles

On construit deux variables de contrôle : Statut_PNL (resp. Statut_PL) vaut 1 si l'établissement est privé non lucratif (resp. privé lucratif) 

In [108]:
FINESS["Statut"].value_counts()

Statut
Privé lucratif        1286
Privé non lucratif    1259
Public                1223
CHU                    206
Name: count, dtype: int64

In [109]:
dummies = pd.get_dummies(Data["Statut"], prefix="Statut")

Data["Statut_PNL"] = dummies["Statut_Privé non lucratif"]
Data["Statut_PL"]  = dummies["Statut_Privé lucratif"]
Data["Statut_CHU"] = dummies["Statut_CHU"]

Data["Statut_PNL"] = Data["Statut_PNL"].astype(int)
Data["Statut_PL"]  = Data["Statut_PL"].astype(int)
Data["Statut_CHU"] = Data["Statut_CHU"].astype(int)

Ajout de variables d'outputs liées aux différentes catégories d'établissement contrôlées ("termes d'interaction") :

In [110]:
for var in outputs:
    Data[f"{var}_PNL"] = Data[var] * Data["Statut_PNL"]
    Data[f"{var}_PL"]  = Data[var] * Data["Statut_PL"]
    Data[f"{var}_CHU"]  = Data[var] * Data["Statut_CHU"]

Pour mieux mesurer le gain d'efficacité lié au développement de la médecine ambulatoire (dans les services MCO), on construit deux variables : "duree_MCO" contient la durée moyenne d'un séjour en hospitalisation complète, et "ambulatoire_MCO" contient la proportion d'hospitalisations partielles (ambulatoire) sur le total des hospitalisations prises en charge par le service :

In [111]:
Data["duree_MCO"] = SYGEN2022['JOU_MCO']/SYGEN2022['SEJHC_MCO']
Data["ambulatoire_MCO"] = SYGEN2022['SEJHP_MCO']/(SYGEN2022['SEJHC_MCO']+SYGEN2022['SEJHP_MCO'])

Data["duree_MCO"] = Data["duree_MCO"].fillna(0)
Data["ambulatoire_MCO"] = Data["ambulatoire_MCO"].fillna(0)

Catégorisation des variables :

In [112]:
inputs = ["lMED", "lIDE", "lAID", "lADMIN"]
outputs = ["lURG", "lMCO", "lSSR", "lPSY","lSEANCES", "lUSLD"]
controles_statut = ["Statut_PNL", "Statut_PL","Statut_CHU"] 
controles_services = ["I_PSY", "I_SSR", "I_URG", "I_USLD","I_SEANCES", "I_MCO"]
controles_sejours = ["duree_MCO","ambulatoire_MCO"]
interactions_statut = [f"{var}_PNL" for var in outputs] + [f"{var}_PL" for var in outputs] + [f"{var}_CHU" for var in outputs]

### Régression : estimation des coefficients de la demande conditionnelle et interprétations

Préparation des données à la régression : 

In [113]:
X_vars = (outputs + controles_statut + controles_services + controles_sejours + interactions_statut)

Data_clean = Data[inputs + X_vars].copy()

# Conversion forcée en numérique
Data_clean = Data_clean.apply(pd.to_numeric, errors="coerce")

# Suppression des lignes inexploitables
Data_clean = Data_clean.dropna()

Data_clean.describe()

,lMED,lIDE,lAID,lADMIN,lURG,lMCO,lSSR,lPSY,lSEANCES,lUSLD,...,lSSR_PL,lPSY_PL,lSEANCES_PL,lUSLD_PL,lURG_CHU,lMCO_CHU,lSSR_CHU,lPSY_CHU,lSEANCES_CHU,lUSLD_CHU
count,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,...,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000,2416.000000
mean,3.218307,3.951007,3.756924,3.379992,2.576282,4.502992,3.352062,1.546142,2.248950,0.497824,...,1.044586,0.426151,0.609101,0.016312,0.386505,0.511042,0.172182,0.154954,0.364993,0.040884
std,1.461779,1.414788,1.336891,1.236667,4.445902,4.539988,3.061020,3.298786,3.822177,1.192215,...,2.381080,1.825980,2.171056,0.224365,1.993570,2.229249,1.017715,1.108425,1.834880,0.384513
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.945910,2.833213,2.833213,2.397895,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,2.944439,3.610918,3.583519,3.135494,0.000000,4.844187,5.117994,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,4.418841,5.003946,4.607658,4.219508,8.412504,9.304354,6.200509,0.000000,5.249398,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,7.323831,7.645398,7.515889,6.857514,11.640412,11.568370,7.921536,10.761492,11.306553,4.919981,...,7.798523,10.141559,11.029699,3.931826,11.640412,11.568370,7.800573,10.328723,11.273551,4.919981


On va donc réaliser une estimation à partir de 2548 observations sur l'année 2022.

Estimation des coefficients :

In [114]:
results = {}
dict_diff = {}

In [115]:
for inp in inputs:
    y = Data_clean[inp]

    X = Data_clean[X_vars]
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit(cov_type="HC1")
    results[inp] = model

In [116]:
for inp, model in results.items():
    for statut in ["PNL", "PL", "CHU"]:
        coef_name = f"Statut_{statut}"

        if coef_name in model.params.index:
            coef = model.params[coef_name]
            pct = (np.exp(coef) - 1) * 100
            dict_diff[(inp, statut)] = pct

In [117]:
dict_diff

{('lMED', 'PNL'): -15.804744210554878,
 ('lMED', 'PL'): -15.730249898068028,
 ('lMED', 'CHU'): 89.2688932838815,
 ('lIDE', 'PNL'): -36.198194747347,
 ('lIDE', 'PL'): -36.61810246388681,
 ('lIDE', 'CHU'): 219.60806316653083,
 ('lAID', 'PNL'): -54.60859360524704,
 ('lAID', 'PL'): -58.1016480717016,
 ('lAID', 'CHU'): 129.5155113996007,
 ('lADMIN', 'PNL'): -17.401492888459813,
 ('lADMIN', 'PL'): -46.787267897946094,
 ('lADMIN', 'CHU'): 37.567613472917614}

| Personnel\Établissement   |    Privé Lucratif     |   Privé non lucratif    |    CHU   |
|:----------|------------:|-------------:|-------------:|
| Personnel administratif    |      -50% |        -22% |    +30%     |
| Aides-soigant.e.s      |      -60% |     -57%  |      +126%   |
| Infirmièr.e.s      |      -42% |       -41% |    +189%      |
| Médecins      |      -20% |     -20%    |       +88%  |

tableau regroupant les différences d'emploi des différentes catégories de personnel à niveau de production égal selon la nature de l'etablissement (privé non lucratif,privé lucratif ou CHU) en prenant comme référence les hopitaux publics petits ou moyens (non CHU).

Contrôles : présence de service d'urgence, de MCO, de psychiatrie, de SSR, d'USLD, et de séances ; durée moyenne de séjour en MCO, part d'ambulatoire dans les hospitalisations en MCO.

Contrôles à rajouter : indice de gravité (hospidiag), indice de recherche/enseignement (hospidiag)

Evaluation des régression : R²

In [118]:
for inp, model in results.items():
    print(f"{inp} : R² = {model.rsquared:.3f} | R² ajusté = {model.rsquared_adj:.3f}")

lMED : R² = 0.844 | R² ajusté = 0.842
lIDE : R² = 0.813 | R² ajusté = 0.810
lAID : R² = 0.748 | R² ajusté = 0.744
lADMIN : R² = 0.801 | R² ajusté = 0.798


Concernant l'influence mesurée du développement de la médecine ambulatoire :

In [119]:
for inp, model in results.items():
    print(f"\n{inp}")
    print(model.params[["duree_MCO", "ambulatoire_MCO"]])


lMED
duree_MCO         -0.001556
ambulatoire_MCO    0.083982
dtype: float64

lIDE
duree_MCO         -0.000779
ambulatoire_MCO   -0.039009
dtype: float64

lAID
duree_MCO          0.000131
ambulatoire_MCO    0.054489
dtype: float64

lADMIN
duree_MCO          0.000692
ambulatoire_MCO    0.039315
dtype: float64


les coefficients associés à la variable "ambulatoire_MCO" sont négatifs, de l'ordre de -0.1 pour chaque catégorie de personnel, une augmentation de la proportion d'hospitalisations partielles permet en moyenne d'économiser du personnel (ou plutot, en moyenne, les établissement ayant le plus développé les soins ambulatoires emploient en moyenne moins de personnel qu'un hôpital petit hôpital représentatif)